## Description

## Import libraries

In [64]:
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client.models import Prefetch, SparseVector, FusionQuery
from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client import QdrantClient
import cohere
from dotenv import load_dotenv
import os
from google import genai

In [8]:
load_dotenv()

True

## RAG pipeline implementation

In [145]:
def get_dense_embedding(dense_model, text):

    return next(dense_model.embed([text])).tolist()

In [146]:
def get_sparse_embedding(sparse_model, text):

    sparse_embedding = next(sparse_model.embed([text]))

    values = sparse_embedding.values.tolist()
    indices = sparse_embedding.indices.tolist()

    return values, indices

In [189]:
def retrieval(query, qdrant_client, dense_model, sparse_model, limit_vector_search=200, limit_keywords_search=200, limit_result=100):

    dense_embedding = get_dense_embedding(dense_model, query)
    sparse_embedding_values, sparse_embedding_indices = get_sparse_embedding(sparse_model, query)


    results = qdrant_client.query_points(
        collection_name="regulations",
        with_payload=True,
        prefetch=[
            Prefetch(
                query=dense_embedding,
                using="dense",
                limit=limit_vector_search,
            ),
            Prefetch(
                query=SparseVector(
                    indices=sparse_embedding_indices,
                    values=sparse_embedding_values
                ),
                using="sparse",
                limit=limit_keywords_search,
            ),
        ],
        query=FusionQuery(fusion="rrf"),
        limit=limit_result
    )


    return results.points

In [179]:
def reranking(query, list_points, cohere_client, top_n=20):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)


    response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=texts,
        top_n=top_n,
    )
    
    reranking_result = [ list_points[item.index] for item in response.results ]


    return reranking_result

In [ ]:
def generate_multi_queries(query, gemini_client, number_generated_queries=4):

    gemini_prompt = f"""
    Generate {number_generated_queries} alternative search queries for the following question.

    The queries should have the same meaning but use different wording.

    Question:
    {query}

    Return only the queries, one per line.
    """

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=gemini_prompt
    )

    return [query] + response.text.split("\n")

In [182]:
def reciprocal_rank_fusion(result_lists, k=60):

    id_point = {}

    for points_list in result_lists:

        for idx, point in enumerate(points_list):

            id = point.id

            if id not in id_point:
                id_point[id] = point
                id_point[id].score = 0

            id_point[id].score += ( 1 / (k + (idx+1)) )


    rrf_result = sorted( list(id_point.items()), key=lambda x: x[1].score, reverse=True )
    rrf_result = list( dict(rrf_result).values() )


    return rrf_result     

In [183]:
def generate_answer(user_query, list_points, gemini_client):

    texts = []

    for point in list_points:

        text = f"""
        Regulation Type: {point.payload['regulation_type']} regulation
        Year: {point.payload['year']}
        Article: {point.payload['article']}
        Chapter: {point.payload['chapter']}

        {point.payload['content']}
        """.strip()

        texts.append(text)

    context = '\n\n'.join(texts)


    prompt = f"""
    You are an expert on FIA Formula 1 Regulations.

    Answer the question using ONLY the provided context.

    Rules:
    - Do not use outside knowledge.
    - If the answer is not available in the context, say:
    "The provided regulations do not contain enough information to answer this question."
    - For every factual statement, provide the corresponding year, regulation type and article number.
    - When comparing regulations from different years, clearly indicate which year and article each statement comes from.
    - If multiple articles are relevant, cite all of them.
    - Be precise and concise.
    - Use bullet points when comparing regulations.

    Context:
    {context}

    Question:
    {user_query}
    """


    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )


    return response.text


In [184]:
def rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client):

    queries = generate_multi_queries(user_query, gemini_client)

    hybrid_search_results_lists = [ retrieval(query, qdrant_client, dense_model, sparse_model) for query in queries]
    
    rrf_result = reciprocal_rank_fusion(hybrid_search_results_lists)

    reranking_result = reranking(user_query, rrf_result, cohere_client)

    final_answer = generate_answer(user_query, reranking_result, gemini_client)


    return final_answer, reranking_result

## Testing retrieve and generation

In [67]:
dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/minicoil-v1"
)

qdrant_client = QdrantClient(path='../data/qdrant_storage')

cohere_client = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))

gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
# 'Please tell me about the Safety Car rules in 2025 year.'  

In [185]:
user_query = 'How have the Safety Car rules changed between 2018 and 2026?'      
answer, retrieved_chunks = rag_pipeline(user_query, gemini_client, qdrant_client, dense_model, sparse_model, cohere_client)

In [186]:
print(answer)

The provided regulations do not contain enough information to answer this question for the year 2018. The earliest regulations provided are from 2019.

However, based on the provided context, the Safety Car rules have undergone several changes between 2019 and 2026:

*   **Initial Position and Deployment of the Safety Car:**
    *   **2019 (sporting regulation, Article 39.2) & 2020 (sporting regulation, Article 39.2):** The safety car would leave the pit lane fifty minutes before the start of the formation lap, take position at the front of the grid until the five-minute signal, then cover a whole lap and take position.
    *   **2021 (sporting regulation, Article 48.2):** The safety car would leave the pit lane prior to the pit lane opening for a reconnaissance lap, take position at the front of the grid until the five-minute signal, then cover a whole lap and take position.
    *   **2026 (sporting regulation, Article B5.10.2):** The Safety Car leaves the grid when the green lights o

In [187]:
for i, point in enumerate(retrieved_chunks, start=1):
    p = point.payload

    print("=" * 120)
    print(f"Result #{i}")
    print(f"Score      : {point.score:.4f}")
    print(f"Year       : {p['year']}")
    print(f"Article    : {p['article']}")
    print(f"Chapter    : {p['chapter']}")
    print("-" * 120)
    print(p['content'])
    print()

Result #1
Score      : 0.0861
Year       : 2026
Article    : B5.13.8
Chapter    : TOTAL TIME CLASSIFIED SESSIONS (TTCS), Safety Car (SC)
------------------------------------------------------------------------------------------------------------------------
If the Safety Car is still deployed at the beginning of the last lap, or is deployed during the last lap, unless the Race Director deems the continued presence of the Safety Car after the end-of-session signal is required, it will enter the Pit Lane at the end of the lap and the F1 Cars must proceed to take the end-of-session signal without overtaking before the Line. In such circumstance, the SC boards and the yellow flags will not be withdrawn but, as the Safety Car is approaching the Pit Entry Road, the orange lights on the Safety Car will be extinguished. This will be the signal to all Competitors and drivers that the Safety Car will be entering the Pit Lane at the end of that lap. The chequered flag will be shown at the Line in

Сформувати 10 питань і оцінити на них знайдені чанки та якість фінальної відповіді